# Evaluación de modelos para BugTriage AI

## Objetivo

Comparar modelos de clasificación zero-shot para determinar cuál resulta más adecuado
para clasificar reportes de bugs de software.

La evaluación se realizará utilizando exactamente los mismos casos de prueba para
todos los modelos.

### Modelos candidatos

1. `facebook/bart-large-mnli`
2. `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`

### Hipótesis

Se espera que el modelo multilingüe pueda presentar un mejor comportamiento sobre
reportes escritos en español, mientras que `facebook/bart-large-mnli` constituye
un modelo de referencia ampliamente utilizado para clasificación zero-shot.

La hipótesis será validada experimentalmente y no se asumirá que un modelo es mejor
antes de realizar las pruebas.

In [ ]:
from transformers import pipeline
import pandas as pd
import time

In [ ]:
CATEGORIAS = [
    "Funcional",
    "UI/UX",
    "Autenticación",
    "Datos",
    "Accesibilidad"
]

CATEGORIAS

['Funcional', 'UI/UX', 'Autenticación', 'Datos', 'Accesibilidad']

## Casos de prueba

Se utilizarán los mismos casos para evaluar todos los modelos candidatos.

El conjunto incluye casos normales y casos potencialmente difíciles:
- texto muy corto;
- lenguaje informal;
- lenguaje técnico/jargon;
- español e inglés;
- mezcla de idiomas;
- reportes ambiguos;
- más de un problema mencionado.

In [ ]:
casos_prueba = [
    {
        "id": 1,
        "texto": "El botón de guardar no funciona.",
        "esperado": "Funcional"
    },
    {
        "id": 2,
        "texto": "La pantalla de login se rompe cuando ingreso mi contraseña.",
        "esperado": "Autenticación"
    },
    {
        "id": 3,
        "texto": "Los nombres de los usuarios aparecen duplicados.",
        "esperado": "Datos"
    },
    {
        "id": 4,
        "texto": "El botón de continuar queda cortado en la versión móvil.",
        "esperado": "UI/UX"
    },
    {
        "id": 5,
        "texto": "El lector de pantalla no anuncia el campo de correo electrónico.",
        "esperado": "Accesibilidad"
    },
    {
        "id": 6,
        "texto": "No anda.",
        "esperado": "Funcional"
    },
    {
        "id": 7,
        "texto": "Después del último deploy rompieron el flow de login.",
        "esperado": "Autenticación"
    },
    {
        "id": 8,
        "texto": "The submit button is broken on mobile.",
        "esperado": "UI/UX"
    },
    {
        "id": 9,
        "texto": "El login tira error después del deploy.",
        "esperado": "Autenticación"
    },
    {
        "id": 10,
        "texto": "Desde el celular no puedo terminar el registro.",
        "esperado": "UI/UX"
    },
    {
        "id": 11,
        "texto": "Che, cada vez que guardo se me borra todo.",
        "esperado": "Datos"
    },
    {
        "id": 12,
        "texto": "El login anda perfecto, pero el lector no me lee el botón.",
        "esperado": "Accesibilidad"
    }
]

df_casos = pd.DataFrame(casos_prueba)

df_casos


,id,texto,esperado
0,1,El botón de guardar no funciona.,Funcional
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación
2,3,Los nombres de los usuarios aparecen duplicados.,Datos
3,4,El botón de continuar queda cortado en la vers...,UI/UX
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad
5,6,No anda.,Funcional
6,7,Después del último deploy rompieron el flow de...,Autenticación
7,8,The submit button is broken on mobile.,UI/UX
8,9,El login tira error después del deploy.,Autenticación
9,10,Desde el celular no puedo terminar el registro.,UI/UX


In [ ]:
casos_dificiles = {
    6: "Muy corto",
    7: "Jargon técnico",
    8: "Otro idioma",
    9: "Mezcla de lenguaje técnico",
    10: "Potencialmente ambiguo",
    11: "Lenguaje informal",
    12: "Dos conceptos en el mismo reporte"
}

casos_dificiles

{6: 'Muy corto',
 7: 'Jargon técnico',
 8: 'Otro idioma',
 9: 'Mezcla de lenguaje técnico',
 10: 'Potencialmente ambiguo',
 11: 'Lenguaje informal',
 12: 'Dos conceptos en el mismo reporte'}

### Casos difíciles

Los casos 6 a 12 permiten observar limitaciones del modelo frente a situaciones
que pueden ser más difíciles que una clasificación directa.

En particular, se incluyen textos muy cortos, terminología utilizada en equipos
de desarrollo, lenguaje informal, inglés, mezcla de idiomas y reportes que
contienen más de un elemento potencialmente relevante.



## PROBAR MODELOS

## Modelo 1 — facebook/bart-large-mnli

Este modelo utiliza Natural Language Inference (NLI) para realizar clasificación
zero-shot.

Se evaluará utilizando las mismas categorías y los mismos casos definidos
anteriormente.

In [ ]:
modelo_1 = "facebook/bart-large-mnli"

inicio = time.perf_counter()

clasificador_1 = pipeline(
    "zero-shot-classification",
    model=modelo_1
)

tiempo_carga_1 = time.perf_counter() - inicio

print("Modelo cargado:", modelo_1)
print(f"Tiempo de carga: {tiempo_carga_1:.2f} segundos")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Modelo cargado: facebook/bart-large-mnli
Tiempo de carga: 55.97 segundos


In [ ]:
resultado_prueba_1 = clasificador_1(
    casos_prueba[0]["texto"],
    CATEGORIAS
)

resultado_prueba_1

{'sequence': 'El botón de guardar no funciona.',
 'labels': ['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional'],
 'scores': [0.26611194014549255,
  0.26195088028907776,
  0.2077663242816925,
  0.19364923238754272,
  0.07052168250083923]}

In [ ]:
type(resultado_prueba_1)


dict

In [ ]:
resultado_prueba_1["labels"]

['UI/UX', 'Autenticación', 'Datos', 'Accesibilidad', 'Funcional']

In [ ]:
resultado_prueba_1["scores"]

[0.26611194014549255,
 0.26195088028907776,
 0.2077663242816925,
 0.19364923238754272,
 0.07052168250083923]

In [13]:
resultados_modelo_1 = []

inicio_total_1 = time.perf_counter()

for caso in casos_prueba:
    inicio = time.perf_counter()

    resultado = clasificador_1(
        caso["texto"],
        CATEGORIAS
    )

    tiempo = time.perf_counter() - inicio

    prediccion = resultado["labels"][0]
    score = resultado["scores"][0]

    resultados_modelo_1.append({
        "id": caso["id"],
        "frase": caso["texto"],
        "esperado": caso["esperado"],
        "prediccion": prediccion,
        "score": score,
        "correcto": prediccion == caso["esperado"],
        "tiempo_segundos": tiempo
    })

tiempo_total_1 = time.perf_counter() - inicio_total_1

df_modelo_1 = pd.DataFrame(resultados_modelo_1)

df_modelo_1

,id,frase,esperado,prediccion,score,correcto,tiempo_segundos
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112,False,1.001640
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Autenticación,0.307510,True,0.896505
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,Datos,0.325560,True,0.879123
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505,False,0.929636
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,Accesibilidad,0.281191,True,0.957054
5,6,No anda.,Funcional,Datos,0.409652,False,0.838082
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505,False,0.869987
7,8,The submit button is broken on mobile.,UI/UX,UI/UX,0.622319,True,0.818239
8,9,El login tira error después del deploy.,Autenticación,Autenticación,0.296833,True,0.866580
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046,False,0.861106


In [14]:
precision_1 = df_modelo_1["correcto"].mean()

print(f"Precisión: {precision_1:.2%}")
print(f"Tiempo total: {tiempo_total_1:.2f} segundos")
print(f"Tiempo promedio: {df_modelo_1['tiempo_segundos'].mean():.2f} segundos")

Precisión: 50.00%
Tiempo total: 10.79 segundos
Tiempo promedio: 0.90 segundos


In [16]:
df_modelo_1.to_csv(
    "resultados_bart_large_mnli.csv",
    index=False
)

In [28]:
df_modelo_1[
    ["id", "frase", "esperado", "prediccion", "score", "correcto"]
]

,id,frase,esperado,prediccion,score,correcto
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112,False
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Autenticación,0.307510,True
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,Datos,0.325560,True
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505,False
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,Accesibilidad,0.281191,True
5,6,No anda.,Funcional,Datos,0.409652,False
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505,False
7,8,The submit button is broken on mobile.,UI/UX,UI/UX,0.622319,True
8,9,El login tira error después del deploy.,Autenticación,Autenticación,0.296833,True
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046,False


In [29]:
print("ERRORES DEL MODELO 1")

df_modelo_1[
    df_modelo_1["correcto"] == False
][
    ["id", "frase", "esperado", "prediccion", "score"]
]

ERRORES DEL MODELO 1


,id,frase,esperado,prediccion,score
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.266112
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Accesibilidad,0.281505
5,6,No anda.,Funcional,Datos,0.409652
6,7,Después del último deploy rompieron el flow de...,Autenticación,UI/UX,0.360505
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.334046
10,11,"Che, cada vez que guardo se me borra todo.",Datos,Autenticación,0.374223


## Modelo 2 — MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7

Este modelo fue seleccionado como candidato multilingüe para evaluar su comportamiento
sobre reportes de bugs escritos principalmente en español.

Al igual que el modelo anterior, se utilizará para clasificación zero-shot mediante
la tarea `zero-shot-classification`.

Se mantendrán exactamente las mismas categorías y los mismos 12 casos de prueba
para que la comparación sea equivalente.

In [17]:
modelo_2 = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

inicio = time.perf_counter()

clasificador_2 = pipeline(
    "zero-shot-classification",
    model=modelo_2
)

tiempo_carga_2 = time.perf_counter() - inicio

print("Modelo cargado:", modelo_2)
print(f"Tiempo de carga: {tiempo_carga_2:.2f} segundos")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

D:\TECNICATURA IA\seminario de actualizacion 2026\proyecto_integrador\TP_Integrador\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Modelo cargado: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Tiempo de carga: 41.32 segundos


In [18]:
resultado_prueba_2 = clasificador_2(
    casos_prueba[0]["texto"],
    CATEGORIAS
)

resultado_prueba_2

{'sequence': 'El botón de guardar no funciona.',
 'labels': ['UI/UX', 'Accesibilidad', 'Datos', 'Autenticación', 'Funcional'],
 'scores': [0.45136383175849915,
  0.2290823608636856,
  0.18632791936397552,
  0.09525997191667557,
  0.03796588256955147]}

In [19]:
type(resultado_prueba_2)


dict

In [20]:
resultado_prueba_2["labels"]

['UI/UX', 'Accesibilidad', 'Datos', 'Autenticación', 'Funcional']

In [21]:
resultado_prueba_2["scores"]

[0.45136383175849915,
 0.2290823608636856,
 0.18632791936397552,
 0.09525997191667557,
 0.03796588256955147]

In [25]:
resultados_modelo_2 = []

inicio_total_2 = time.perf_counter()

for caso in casos_prueba:
    inicio = time.perf_counter()

    resultado = clasificador_2(
        caso["texto"],
        CATEGORIAS
    )

    tiempo = time.perf_counter() - inicio

    prediccion = resultado["labels"][0]
    score = resultado["scores"][0]

    resultados_modelo_2.append({
        "id": caso["id"],
        "frase": caso["texto"],
        "esperado": caso["esperado"],
        "prediccion": prediccion,
        "score": score,
        "correcto": prediccion == caso["esperado"],
        "tiempo_segundos": tiempo
    })

tiempo_total_2 = time.perf_counter() - inicio_total_2

df_modelo_2 = pd.DataFrame(resultados_modelo_2)

df_modelo_2

,id,frase,esperado,prediccion,score,correcto,tiempo_segundos
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.451364,False,4.130934
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Accesibilidad,0.709738,False,4.404814
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,UI/UX,0.364299,False,3.948743
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Funcional,0.311254,False,4.310154
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,UI/UX,0.600210,False,4.473136
5,6,No anda.,Funcional,Accesibilidad,0.281979,False,3.688059
6,7,Después del último deploy rompieron el flow de...,Autenticación,Accesibilidad,0.614365,False,4.089429
7,8,The submit button is broken on mobile.,UI/UX,Accesibilidad,0.537123,False,4.181588
8,9,El login tira error después del deploy.,Autenticación,Accesibilidad,0.694260,False,3.952380
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.333061,False,4.395223


In [26]:
precision_2 = df_modelo_2["correcto"].mean()

print(f"Precisión: {precision_2:.2%}")
print(f"Tiempo total: {tiempo_total_2:.2f} segundos")
print(f"Tiempo promedio: {df_modelo_2['tiempo_segundos'].mean():.2f} segundos")

Precisión: 16.67%
Tiempo total: 49.82 segundos
Tiempo promedio: 4.15 segundos


In [27]:
df_modelo_2.to_csv(
    "resultados_mdeberta_multilingual.csv",
    index=False
)

In [30]:
print("ERRORES DEL MODELO 2")

df_modelo_2[
    df_modelo_2["correcto"] == False
][
    ["id", "frase", "esperado", "prediccion", "score"]
]

ERRORES DEL MODELO 2


,id,frase,esperado,prediccion,score
0,1,El botón de guardar no funciona.,Funcional,UI/UX,0.451364
1,2,La pantalla de login se rompe cuando ingreso m...,Autenticación,Accesibilidad,0.709738
2,3,Los nombres de los usuarios aparecen duplicados.,Datos,UI/UX,0.364299
3,4,El botón de continuar queda cortado en la vers...,UI/UX,Funcional,0.311254
4,5,El lector de pantalla no anuncia el campo de c...,Accesibilidad,UI/UX,0.600210
5,6,No anda.,Funcional,Accesibilidad,0.281979
6,7,Después del último deploy rompieron el flow de...,Autenticación,Accesibilidad,0.614365
7,8,The submit button is broken on mobile.,UI/UX,Accesibilidad,0.537123
8,9,El login tira error después del deploy.,Autenticación,Accesibilidad,0.694260
9,10,Desde el celular no puedo terminar el registro.,UI/UX,Datos,0.333061


In [31]:
comparacion = pd.DataFrame({
    "Modelo": [
        modelo_1,
        modelo_2
    ],
    "Precisión": [
        precision_1,
        precision_2
    ],
    "Tiempo de carga (s)": [
        tiempo_carga_1,
        tiempo_carga_2
    ],
    "Tiempo total (s)": [
        tiempo_total_1,
        tiempo_total_2
    ],
    "Tiempo promedio por caso (s)": [
        df_modelo_1["tiempo_segundos"].mean(),
        df_modelo_2["tiempo_segundos"].mean()
    ]
})

comparacion

,Modelo,Precisión,Tiempo de carga (s),Tiempo total (s),Tiempo promedio por caso (s)
0,facebook/bart-large-mnli,0.500000,55.974195,10.790579,0.899189
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,0.166667,41.322570,49.819124,4.151561


In [32]:
comparacion.sort_values(
    by="Precisión",
    ascending=False
)

,Modelo,Precisión,Tiempo de carga (s),Tiempo total (s),Tiempo promedio por caso (s)
0,facebook/bart-large-mnli,0.500000,55.974195,10.790579,0.899189
1,MoritzLaurer/mDeBERTa-v3-base-xnli-multilingua...,0.166667,41.322570,49.819124,4.151561


## Análisis preliminar de resultados

El benchmark mostró una diferencia significativa entre los dos modelos evaluados.

`facebook/bart-large-mnli` obtuvo una precisión del 50,00% (6 de 12 casos),
mientras que `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`
obtuvo una precisión del 16,67% (2 de 12 casos).

Además, BART presentó un tiempo promedio de inferencia de 0,90 segundos
por caso, frente a 4,15 segundos del modelo multilingüe.

Por lo tanto, para este conjunto de prueba y bajo las condiciones del
experimento, BART presenta el mejor equilibrio entre precisión y tiempo
de inferencia.

Sin embargo, los resultados no permiten afirmar que BART sea superior
en términos generales. El conjunto utilizado contiene solamente 12 casos
y funciona como benchmark exploratorio para seleccionar un modelo candidato.

Los errores observados se concentran especialmente en reportes muy cortos,
ambiguos, informales o con terminología técnica. Esto muestra una limitación
importante de la clasificación zero-shot: la predicción depende de la
relación semántica entre el texto y las etiquetas proporcionadas, y no de
un modelo entrenado específicamente para las categorías de BugTriage AI.